# 03 — Mini fusion smoke (CPU; bukan evaluasi tesis)

Jalankan empat cell kode berurutan di runtime **CPU**. Notebook ini membaca
cohort mini berpasangan dari folder Drive P2, memverifikasi checksum, lalu
menjalankan satu langkah optimisasi B0 image-only, E1 panjang+arah, dan E2
+IAT. **One-batch loss bukan metrik perbandingan akurasi**. Tidak melatih model
sampai konvergen dan tidak mengunggah/menimpa artefak Drive.


In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from pathlib import Path
import hashlib, io, json, subprocess, sys, tempfile

creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
drive = build('drive', 'v3', credentials=creds, cache_discovery=False)
WORK = Path(tempfile.mkdtemp(prefix='mini_fusion_cpu_'))
COHORT = WORK / 'cohort'
COHORT.mkdir()
print('Workspace:', WORK)


In [ ]:
FILES = {
    'summary.json': '1BtZfv8N5lEXPuoXTYtzhdqGN07s5a5_m',
    'manifest.jsonl': '1VuPqylxOP5diJlPepYDE4-V90aI7PbfO',
    'mini_paired_cohort_NOT_THESIS_EVAL.npz': '1cQswB7PMYsBjyWLUb01bQU_wmSwiIJtI',
}
P2_FOLDER_ID = '1vQyL66nIjIazvU3jtiCXaX8pdNEa34pJ'
for name, file_id in FILES.items():
    meta = drive.files().get(fileId=file_id, fields='id,name,size,parents', supportsAllDrives=True).execute()
    assert meta['name'] == name and P2_FOLDER_ID in meta['parents']
    assert int(meta['size']) < 1024 * 1024
    request = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    with (COHORT / name).open('wb') as output:
        downloader = MediaIoBaseDownload(output, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(name, meta['size'], 'bytes')
summary = json.loads((COHORT/'summary.json').read_text())
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
assert sha256(COHORT/'mini_paired_cohort_NOT_THESIS_EVAL.npz') == summary['outputs']['npz_sha256']
assert sha256(COHORT/'manifest.jsonl') == summary['outputs']['manifest_sha256']
assert summary['status'] == 'FEASIBILITY_ONLY_NOT_THESIS_EVALUATION'
print('Checksum OK; 564 candidates, not a thesis evaluation.')


In [ ]:
REPO = WORK / 'Open-Detect'
subprocess.run(['git', 'clone', '--filter=blob:none', '--single-branch',
                '--branch', 'codex/temporal-fusion',
                'https://github.com/HaikalE/Open-Detect.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'df64785'], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short=7', 'HEAD'], text=True).strip()
assert head == 'df64785', head
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_temporal_fusion',
                'tests.test_mini_cohort', '-q'], cwd=REPO, check=True)
OUT = WORK / 'mini_fusion_smoke_cpu.json'
subprocess.run([sys.executable, '-m', 'pilot.smoke_mini_fusion', '--cohort-dir', str(COHORT),
                '--output', str(OUT)], cwd=REPO, check=True)
print('Smoke selesai; bukan training atau skor tesis:', OUT)


In [ ]:
result = json.loads(OUT.read_text())
assert result['status'] == 'ENGINEERING_ONLY_NOT_EVALUATION'
print('Arms:', list(result['arms']))
print('Checksum cohort:', result['cohort_npz_sha256'])
print('Jangan bandingkan one-batch loss sebagai performa model.')
